In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
files = glob('IndicTTS_*/data/*.parquet')
len(files)

44

In [3]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': df['text'].iloc[i],
                'speaker': f"{base}_{df['gender'].iloc[i]}"
            })
        
    return data

In [4]:
data = multiprocessing(files, loop, cores = min(10, len(files)))

100%|██████████| 4/4 [25:16<00:00, 379.06s/it]


In [5]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'IndicTTS_Telugu_audio/IndicTTS_Telugu-data-train-00003-of-00004_0.mp3',
 'text': 'సూర్యుడి మధ్యరేఖకు సమీపంలో ఉండే మచ్చ, సూర్యుడి ధృవాలకు, దగ్గిరగా ఉండే మచ్చకన్నా, శీఘ్రంగా చుట్టూ తిరిగివస్తుంది, దీనినిబట్టి, సూర్యుడు, భూమిలాగా, ఘనగోళం కాదనీ, సూర్యగోళంలోని, వివిధ ప్రాంతాలు, వేరు వేరు వేగాలతో తిరుగుతున్నాయనీ, తెలుస్తుంది.',
 'speaker': 'IndicTTS_Telugu_audio_1'}

In [6]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'IndicTTS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 30.70ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 5.63MB / 5.63MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 5.63MB / 5.63MB,  0.00B/s  
New Data Upload: 100%|██████████| 3.33MB / 3.33MB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.57 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/345236f0950769fd6e4450aeec62c97f1bc6f50a', commit_message='Upload dataset', commit_description='', oid='345236f0950769fd6e4450aeec62c97f1bc6f50a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [8]:
audio_files = [d['audio_filename'] for d in data]

with open('IndicTTS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
folders = glob('IndicTTS_*_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

IndicTTS_Tamil_audio
IndicTTS_Bengali_audio_neucodec
IndicTTS_Malayalam_audio
IndicTTS_Punjabi_audio_neucodec
IndicTTS_Malayalam_audio_neucodec
IndicTTS_Telugu_audio
IndicTTS_Bengali_audio
IndicTTS_Punjabi_audio
IndicTTS_Tamil_audio_neucodec
IndicTTS_Telugu_audio_neucodec


In [11]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('IndicTTS_*_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  96%|█████████▌| 14.3MB / 14.9MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 14.9MB / 14.9MB, 1.82MB/s  
Processing Files (1 / 1): 100%|██████████| 14.9MB / 14.9MB, 1.57MB/s  
New Data Upload: 100%|██████████| 14.9MB / 14.9MB, 1.57MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  98%|█████████▊| 15.2MB / 15.5MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 15.5MB / 15.5MB, 1.16MB/s  
Processing Files (1 / 1): 100%|██████████| 15.5MB / 15.5MB,  582kB/s  
New Data Upload: 100%|██████████| 14.2MB / 14.2MB,  582kB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   6%|▌         | 46.1MB /  802MB,   ???B/s  
Processing Files (0 / 1):  23%|██▎       |  187MB /  802MB,  704MB/s  
Processing Files (0 / 1):  39%|███▊      |  311MB /  802MB,  662MB/s  
Processing Files (0 / 1):  52%|█████